# SCOPE Position Feature EDA Final

This notebook measures where content structures appear in cleaned main-page content and evaluates whether those measurements are ready for later econometric modeling.

**Boundary:** descriptive associations among sources already surfaced in the audit. This notebook does not fit a new LPM or logistic regression, does not use answer text or ranking variables, and makes no causal claim.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.econometrics_eda_v2.position_feature_eda import run_position_feature_eda

OUT = ROOT / 'outputs/position_feature_eda_final_20260731'
result = run_position_feature_eda(output_dir=OUT)
result

{'status': 'position_feature_eda_ready_for_model_planning',
 'version': 'position_feature_eda_v1_20260731',
 'rows': 5758,
 'unique_urls': 2881,
 'domains': 586,
 'citation_rate': 0.3287599861062869,
 'new_regression_estimated': False,
 'output_dir': '/Volumes/ExtremeSD/Metier/Research/CiteScope-content-audit/outputs/position_feature_eda_final_20260731',
 'static_report': '/Volumes/ExtremeSD/Metier/Research/CiteScope-content-audit/outputs/position_feature_eda_final_20260731/frontend/position_feature_eda_report.html',
 'all_validation_checks_passed': True}

## 1. Sample and extraction status

The row-level estimand remains citation conditional on a source being surfaced. URL-level coverage is reported separately so repeated source appearances are not described as additional webpages.

In [2]:
rows = pd.read_parquet(OUT / 'data/scope_condo_eda_ready_with_position_features.parquet')
coverage = pd.read_csv(OUT / 'tables/position_feature_coverage.csv')
missingness = pd.read_csv(OUT / 'tables/position_feature_missingness.csv')
sample_summary = pd.DataFrame([{
    'surfaced_rows': len(rows),
    'unique_urls': rows.normalized_url.nunique(),
    'unique_domains': rows.source_root_domain.nunique(),
    'unique_prompts': rows.prompt_id.nunique(),
    'citation_rate': rows.cited.mean(),
}])
display(sample_summary)
display(missingness)

,surfaced_rows,unique_urls,unique_domains,unique_prompts,citation_rate
0,5758,2881,586,498,0.32876


,position_extraction_status,pages,domains,page_share
0,main_content_parse_failed,111,35,0.038528
1,measured,2770,565,0.961472


## 2. Feature audit and coverage

Baseline features are not automatically repeated. The audit distinguishes previously available variables from genuinely new position, intensity, density, and interaction measurements.

In [3]:
audit = pd.read_csv(OUT / 'tables/position_feature_audit.csv')
display(audit)
fig = px.bar(
    coverage.sort_values('percentage_of_eligible_pages'),
    x='percentage_of_eligible_pages', y='feature', orientation='h',
    text='pages_with_feature',
    title=f'Feature coverage among eligible pages (n={int(coverage.applicable_page_count.max()):,})',
)
fig.update_xaxes(tickformat='.0%')
fig.show()
display(coverage)

,feature_name,feature_group,feature_definition,already_exists,previously_analyzed,needs_extraction,newly_extracted,include_in_new_eda,reason,extraction_success_rate,missing_rate,applicable_page_count,applicable_domain_count,percentage_of_eligible_pages,fixed_effect_readiness
0,has_direct_answer,answerability_presence,At least one conservative direct-answer phrase...,False,False,True,True,True,New position/intensity measure or required pre...,0.961472,0.038528,2770.0,565.0,0.031408,Weak within-domain variation
1,has_question_heading,answerability_presence,Has question heading.,False,False,True,True,False,Registry concept not extracted in this version...,0.961472,0.038528,2770.0,565.0,0.196029,Ready
2,has_definition_block,answerability_presence,At least one definition list or explicit defin...,False,False,True,True,True,New position/intensity measure or required pre...,0.961472,0.038528,2770.0,565.0,0.409747,Ready
3,has_faq,answerability_presence,Visible FAQ heading/structure or FAQPage marku...,False,False,True,True,True,New position/intensity measure or required pre...,0.961472,0.038528,2770.0,565.0,0.133213,Ready
4,direct_answer_count,answerability_intensity,Direct answer count.,False,False,True,True,True,New position/intensity measure or required pre...,0.961472,0.038528,2770.0,565.0,0.031408,Weak within-domain variation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,domain,taxonomy_control,Domain.,False,False,False,False,False,"Existing identifier, denominator, grouping var...",NaN,NaN,NaN,NaN,NaN,NaN
95,prompt_id,taxonomy_control,Prompt id.,True,False,False,False,False,"Existing identifier, denominator, grouping var...",NaN,NaN,NaN,NaN,NaN,NaN
96,page_id,taxonomy_control,Page id.,False,False,True,True,False,"Existing identifier, denominator, grouping var...",NaN,NaN,NaN,NaN,NaN,NaN
97,word_count,taxonomy_control,Word count.,True,False,False,False,False,"Existing identifier, denominator, grouping var...",NaN,NaN,NaN,NaN,NaN,NaN


,feature,presence_feature,already_present_in_source_data,newly_extracted,extraction_success_rate,missing_rate,applicable_page_count,pages_with_feature,percentage_of_eligible_pages,applicable_domain_count,domains_with_feature,page_types_with_feature,citation_rate_when_present,citation_rate_when_absent,reason
0,table,has_table,True,False,0.961472,0.038528,2770,438,0.158123,565,106,11,0.338446,0.320930,Position-bearing feature included in new EDA.
1,list,has_bullets,False,True,0.961472,0.038528,2770,1292,0.466426,565,260,12,0.300394,0.344396,Position-bearing feature included in new EDA.
2,faq,has_faq,False,True,0.961472,0.038528,2770,369,0.133213,565,86,9,0.314322,0.325680,Position-bearing feature included in new EDA.
3,direct_answer,has_direct_answer,False,True,0.961472,0.038528,2770,87,0.031408,565,27,5,0.420168,0.319696,Position-bearing feature included in new EDA.
4,definition,has_definition_block,False,True,0.961472,0.038528,2770,1135,0.409747,565,284,12,0.350570,0.302601,Position-bearing feature included in new EDA.
5,comparison,has_comparison,False,True,0.961472,0.038528,2770,570,0.205776,565,152,11,0.370532,0.308588,Position-bearing feature included in new EDA.
6,steps,has_steps,False,True,0.961472,0.038528,2770,387,0.139711,565,163,10,0.351259,0.318900,Position-bearing feature included in new EDA.
7,numeric_evidence,has_numeric_evidence,False,True,0.961472,0.038528,2770,1426,0.514801,565,286,13,0.299964,0.348854,Position-bearing feature included in new EDA.
8,external_citation,has_external_sources,False,True,0.961472,0.038528,2770,826,0.298195,565,202,12,0.320513,0.325655,Position-bearing feature included in new EDA.
9,question_heading,has_question_heading,False,True,0.961472,0.038528,2770,543,0.196029,565,153,11,0.351938,0.315489,Position-bearing feature included in new EDA.


## 3. Position distributions and citation rates

`No feature` is distinct from Q1. Position zero means the feature begins at the start of main content. Continuous comparisons are conditional on the feature being present.

In [4]:
distribution = pd.read_csv(OUT / 'tables/position_feature_distribution_summary.csv')
citation = pd.read_csv(OUT / 'tables/citation_rate_by_feature_position.csv')
display(distribution)

for feature in ['table', 'faq', 'direct_answer', 'definition', 'comparison']:
    chart = citation[
        citation.feature.eq(feature)
        & citation.grouping.eq('quartile')
        & citation.sample_scope.eq('full_sample')
    ].sort_values('category_order')
    fig = px.bar(
        chart, x='category', y='citation_rate', text='n_observations',
        error_y=chart.ci_high - chart.citation_rate,
        error_y_minus=chart.citation_rate - chart.ci_low,
        title=f'Citation rate by {feature.replace("_", " ")} position (n={int(chart.n_observations.sum()):,})',
    )
    fig.update_yaxes(tickformat='.0%')
    fig.show()

,feature,position_feature,n_present_rows,n_present_pages,median_position_ratio,q1_position_ratio,q3_position_ratio,mean_position_ratio,minimum_position_ratio,maximum_position_ratio
0,table,first_table_position_ratio,978,438,0.137752,0.051864,0.323171,0.230561,0.0,0.960086
1,list,first_list_position_ratio,2540,1292,0.176707,0.041926,0.445638,0.277511,0.0,0.993103
2,faq,faq_start_position_ratio,789,369,0.793984,0.678922,0.920000,0.751667,0.0,0.985106
3,direct_answer,direct_answer_position_ratio,238,87,0.721931,0.558824,0.738347,0.612398,0.0,0.993815
4,definition,first_definition_position_ratio,2456,1135,0.086528,0.026853,0.273088,0.200744,0.0,0.998427
5,comparison,first_comparison_position_ratio,1371,570,0.327660,0.123581,0.642544,0.393907,0.0,0.988617
6,steps,first_steps_position_ratio,874,387,0.371814,0.105079,0.669425,0.412225,0.0,0.998313
7,numeric_evidence,first_numeric_evidence_position_ratio,2787,1426,0.085496,0.024390,0.238197,0.169923,0.0,0.981781
8,external_citation,first_external_citation_position_ratio,1716,826,0.388485,0.120968,0.881745,0.475039,0.0,0.999404
9,question_heading,first_question_heading_position_ratio,1290,543,0.474194,0.110942,0.860043,0.481772,0.0,0.994032


## 4. Cited versus not-cited positions

Means and medians below use only pages containing the relevant feature. Skewed position distributions make medians especially important.

In [5]:
outcome = pd.read_csv(OUT / 'tables/position_feature_outcome_comparison.csv')
display(outcome)

,feature,position_feature,cited_mean,not_cited_mean,cited_median,not_cited_median,difference_in_means,difference_in_medians,cited_n,not_cited_n
0,table,first_table_position_ratio,0.268239,0.211285,0.178317,0.113833,0.056954,0.064484,331,647
1,list,first_list_position_ratio,0.291390,0.271552,0.216438,0.149254,0.019838,0.067185,763,1777
2,faq,faq_start_position_ratio,0.766356,0.744933,0.817593,0.790698,0.021422,0.026895,248,541
3,direct_answer,direct_answer_position_ratio,0.584160,0.632860,0.721030,0.722411,-0.048700,-0.001381,100,138
4,definition,first_definition_position_ratio,0.181119,0.211338,0.084135,0.087576,-0.030219,-0.003442,861,1595
5,comparison,first_comparison_position_ratio,0.413766,0.382218,0.363221,0.322344,0.031548,0.040877,508,863
6,steps,first_steps_position_ratio,0.430134,0.402529,0.386473,0.368559,0.027605,0.017914,307,567
7,numeric_evidence,first_numeric_evidence_position_ratio,0.180991,0.165181,0.084440,0.086747,0.015810,-0.002307,836,1951
8,external_citation,first_external_citation_position_ratio,0.478955,0.473191,0.402687,0.351703,0.005764,0.050984,550,1166
9,question_heading,first_question_heading_position_ratio,0.456231,0.495642,0.416037,0.512131,-0.039411,-0.096093,454,836


## 5. Domain fixed-effect readiness

Readiness requires position variation and citation-outcome variation within the same domains. The thresholds are documented in the table and are planning rules, not statistical significance tests.

In [6]:
within = pd.read_csv(OUT / 'tables/position_feature_within_domain_diagnostics.csv')
fig = px.scatter(
    within, x='within_domain_standard_deviation',
    y='domains_with_both_position_and_outcome_variation',
    size='informative_observations', color='fixed_effect_readiness',
    hover_name='feature', title=f'Domain fixed-effect readiness (n={len(rows):,} surfaced rows)',
)
fig.show()
display(within)

,feature,position_feature,total_domains,domains_with_at_least_two_pages,domains_containing_feature,domains_with_presence_variation,domains_with_position_ratio_variation,domains_with_position_quartile_variation,domains_with_citation_outcome_variation,domains_with_both_position_and_outcome_variation,informative_observations,share_of_singleton_domains,overall_standard_deviation,between_domain_standard_deviation,within_domain_standard_deviation,within_domain_sd_near_zero,fixed_effect_readiness,readiness_threshold
0,table,first_table_position_ratio,586,244,106,44,38,15,210,33,829,0.583618,0.236159,0.271036,0.114681,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
1,list,first_list_position_ratio,586,244,260,70,106,68,210,83,2130,0.583618,0.276465,0.256706,0.171757,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
2,faq,faq_start_position_ratio,586,244,86,31,40,21,210,32,609,0.583618,0.214860,0.194058,0.146857,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
3,direct_answer,direct_answer_position_ratio,586,244,27,17,7,3,210,7,205,0.583618,0.290373,0.278444,0.075662,False,Weak within-domain variation,"Ready: >=30 domains, >=300 rows, within SD >=0..."
4,definition,first_definition_position_ratio,586,244,284,81,114,64,210,93,2005,0.583618,0.258363,0.223710,0.154546,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
5,comparison,first_comparison_position_ratio,586,244,152,72,63,43,210,56,1183,0.583618,0.305311,0.254347,0.226375,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
6,steps,first_steps_position_ratio,586,244,163,92,54,37,210,46,639,0.583618,0.316946,0.286039,0.201874,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
7,numeric_evidence,first_numeric_evidence_position_ratio,586,244,286,91,119,72,210,89,2345,0.583618,0.208289,0.232511,0.141039,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
8,external_citation,first_external_citation_position_ratio,586,244,202,78,91,55,210,74,1434,0.583618,0.374178,0.350349,0.236185,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."
9,question_heading,first_question_heading_position_ratio,586,244,153,61,64,44,210,53,1042,0.583618,0.371331,0.303622,0.254059,False,Ready,"Ready: >=30 domains, >=300 rows, within SD >=0..."


## 6. Sparse cells, taxonomy confounding, and page length

Four quartiles are retained only when observed cells support them. Taxonomy cross-tabs use normalized percentages. Relative and absolute positions are compared with page length because the same token index can imply a different relative location on short and long pages.

In [7]:
sparse = pd.read_csv(OUT / 'tables/position_feature_sparse_cell_diagnostics.csv')
taxonomy = pd.read_csv(OUT / 'tables/position_feature_taxonomy_crosstab.csv')
page_length = pd.read_csv(OUT / 'tables/position_feature_page_length_relationship.csv')
display(sparse)
display(taxonomy.head(50))
display(page_length)

,category,n_observations,n_unique_pages,cited_count,not_cited_count,citation_rate,ci_low,ci_high,feature,position_feature,grouping,sample_scope,overall_citation_rate,citation_rate_difference,category_order,sparse_n_lt_20,sparse_cited_lt_5,sparse_not_cited_lt_5,sparse_flag,recommended_grouping
0,No feature,4515,2332,1449,3066,0.320930,0.307470,0.334695,table,first_table_position_ratio,quartile,full_sample,0.32876,-0.007830,0,False,False,False,False,"Q1, Q2, Q3, Q4"
1,Q1,622,309,186,436,0.299035,0.264378,0.336160,table,first_table_position_ratio,quartile,full_sample,0.32876,-0.029725,1,False,False,False,False,"Q1, Q2, Q3, Q4"
2,Q2,207,78,84,123,0.405797,0.341210,0.473817,table,first_table_position_ratio,quartile,full_sample,0.32876,0.077037,2,False,False,False,False,"Q1, Q2, Q3, Q4"
3,Q3,119,37,54,65,0.453782,0.367175,0.543279,table,first_table_position_ratio,quartile,full_sample,0.32876,0.125022,3,False,False,False,False,"Q1, Q2, Q3, Q4"
4,Q4,30,14,7,23,0.233333,0.117924,0.409283,table,first_table_position_ratio,quartile,full_sample,0.32876,-0.095427,4,False,False,False,False,"Q1, Q2, Q3, Q4"
5,Unmeasured,265,111,113,152,0.426415,0.368340,0.486593,table,first_table_position_ratio,quartile,full_sample,0.32876,0.097655,6,False,False,False,False,"Q1, Q2, Q3, Q4"
6,No feature,2953,1478,1017,1936,0.344396,0.327469,0.361726,list,first_list_position_ratio,quartile,full_sample,0.32876,0.015636,0,False,False,False,False,"Q1, Q2, Q3, Q4"
7,Q1,1458,756,413,1045,0.283265,0.260729,0.306939,list,first_list_position_ratio,quartile,full_sample,0.32876,-0.045495,1,False,False,False,False,"Q1, Q2, Q3, Q4"
8,Q2,526,247,183,343,0.347909,0.308440,0.389583,list,first_list_position_ratio,quartile,full_sample,0.32876,0.019149,2,False,False,False,False,"Q1, Q2, Q3, Q4"
9,Q3,336,169,107,229,0.318452,0.270931,0.370078,list,first_list_position_ratio,quartile,full_sample,0.32876,-0.010308,3,False,False,False,False,"Q1, Q2, Q3, Q4"


,taxonomy_value,position_category,n_observations,cited_count,unique_pages,row_percent_within_position,row_percent_within_taxonomy,citation_rate,feature,taxonomy_dimension
0,commercial_product_or_service,Q1,69,17,46,0.110932,0.704082,0.246377,table,page_type
1,commercial_product_or_service,Q2,16,3,8,0.077295,0.163265,0.187500,table,page_type
2,commercial_product_or_service,Q3,13,3,7,0.109244,0.132653,0.230769,table,page_type
3,comparison_or_review,Q1,109,55,47,0.175241,0.534314,0.504587,table,page_type
4,comparison_or_review,Q2,89,42,25,0.429952,0.436275,0.471910,table,page_type
5,comparison_or_review,Q3,6,3,4,0.050420,0.029412,0.500000,table,page_type
6,contact_or_location,Q1,2,0,2,0.003215,0.500000,0.000000,table,page_type
7,contact_or_location,Q2,2,0,1,0.009662,0.500000,0.000000,table,page_type
8,directory_or_listing,Q1,268,57,163,0.430868,0.875817,0.212687,table,page_type
9,directory_or_listing,Q2,32,12,10,0.154589,0.104575,0.375000,table,page_type


,feature,page_length_group,n_observations,n_unique_pages,median_word_count,median_position_ratio,median_absolute_token_position,citation_rate,dominant_position_quartile
0,table,Short,67,35,501.0,0.250000,39.0,0.223881,Q2
1,table,Medium,295,126,1084.0,0.264064,218.0,0.355932,Q1
2,table,Long,329,149,1756.0,0.099080,140.0,0.343465,Q1
3,table,Very long,287,128,3939.0,0.069906,246.0,0.341463,Q1
4,list,Short,286,172,570.5,0.326194,97.0,0.318182,Q1
5,list,Medium,684,259,1084.0,0.304491,230.0,0.406433,Q1
6,list,Long,712,403,1992.0,0.101950,141.0,0.248596,Q1
7,list,Very long,858,458,3725.0,0.088976,239.0,0.252914,Q1
8,faq,Short,78,38,500.0,0.738095,248.0,0.435897,Q4
9,faq,Medium,214,92,1084.0,0.791083,563.0,0.294393,Q4


## 7. Correlation and redundancy

Continuous measures use Spearman correlations. Binary presence indicators use phi correlations. Arbitrary category codes are never treated as continuous values.

In [8]:
associations = pd.read_csv(OUT / 'tables/position_feature_associations.csv')
high_pairs = pd.read_csv(OUT / 'tables/position_feature_high_correlation_pairs.csv')
categorical_associations = pd.read_csv(OUT / 'tables/position_feature_categorical_association.csv')
display(high_pairs)
for association_type in associations.association_type.unique():
    subset = associations[associations.association_type.eq(association_type)]
    pivot = subset.pivot(index='feature_a', columns='feature_b', values='association')
    px.imshow(
        pivot, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', aspect='auto',
        title=f'{association_type.replace("_", " ").title()} association matrix',
    ).show()
display(categorical_associations)

,feature_a,feature_b,association,association_type,warning
0,direct_answer_position_ratio,first_numeric_evidence_position_ratio,-0.746126,spearman,High absolute Spearman correlation; avoid simu...
1,table_count,table_count_per_1000_words,0.996496,spearman,High absolute Spearman correlation; avoid simu...
2,list_count,list_item_density,0.938661,spearman,High absolute Spearman correlation; avoid simu...


,feature,position_category_feature,categorical_variable,association_type,association,n_pairwise_complete,warning
0,table,first_table_position_quartile,page_type,bias_corrected_cramers_v,0.159099,5758,NaN
1,table,first_table_position_quartile,source_type,bias_corrected_cramers_v,0.245160,5758,NaN
2,table,first_table_position_quartile,intent,bias_corrected_cramers_v,0.074185,5758,NaN
3,list,first_list_position_quartile,page_type,bias_corrected_cramers_v,0.174873,5758,NaN
4,list,first_list_position_quartile,source_type,bias_corrected_cramers_v,0.230248,5758,NaN
5,list,first_list_position_quartile,intent,bias_corrected_cramers_v,0.108447,5758,NaN
6,faq,faq_start_position_quartile,page_type,bias_corrected_cramers_v,0.124661,5758,NaN
7,faq,faq_start_position_quartile,source_type,bias_corrected_cramers_v,0.129132,5758,NaN
8,faq,faq_start_position_quartile,intent,bias_corrected_cramers_v,0.086736,5758,NaN
9,direct_answer,direct_answer_position_quartile,page_type,bias_corrected_cramers_v,0.162000,5758,NaN


## 8. Manual stored-evidence QA and validation checks

The review sample contains 10 detected and 10 measured-negative pages for each major feature. It is based on stored cleaned HTML/Markdown; live-page drift remains possible.

In [9]:
manual = pd.read_csv(OUT / 'tables/position_feature_manual_validation.csv')
checks = pd.read_csv(OUT / 'tables/position_feature_validation_checks.csv')
display(manual.groupby(['feature', 'review_stratum', 'manual_validation_result']).size().reset_index(name='n'))
display(manual)
display(checks)
assert checks.passed.astype(bool).all()

,feature,review_stratum,manual_validation_result,n
0,comparison,detected_positive,true_positive,10
1,comparison,measured_negative,true_negative_in_stored_snapshot,10
2,definition,detected_positive,true_positive,10
3,definition,measured_negative,true_negative_in_stored_snapshot,10
4,direct_answer,detected_positive,true_positive,10
5,direct_answer,measured_negative,true_negative_in_stored_snapshot,10
6,faq,detected_positive,true_positive,10
7,faq,measured_negative,true_negative_in_stored_snapshot,10
8,table,detected_positive,true_positive,10
9,table,measured_negative,true_negative_in_stored_snapshot,10


,feature,review_stratum,normalized_url,source_url,domain,detected_value,position_ratio,stored_evidence,manual_validation_result,review_note,validation_source
0,table,detected_positive,https://amazingproperties.org/search/bangkok/p...,https://amazingproperties.org/search/bangkok/p...,amazingproperties.org,1,0.581554,"[""Penthouse Segment Space Monthly Rental (THB)...",true_positive,Stored cleaned-main-content evidence visibly s...,stored_cleaned_main_content_html_or_markdown
1,table,detected_positive,https://apthai.com/th/blog/homestory/high-rise...,https://www.apthai.com/th/blog/homestory/high-...,apthai.com,1,0.050149,"[""รายชื่อคอนโด High Rise พื้นที่ใช้สอยและรถไฟฟ...",true_positive,Stored cleaned-main-content evidence visibly s...,stored_cleaned_main_content_html_or_markdown
2,table,detected_positive,https://asterofasia.com/blog/thailand-real-est...,https://asterofasia.com/blog/thailand-real-est...,asterofasia.com,1,0.497382,"[""Parameter Freehold Condo Leasehold Villa Via...",true_positive,Stored cleaned-main-content evidence visibly s...,stored_cleaned_main_content_html_or_markdown
3,table,detected_positive,https://avacasa.life/guides/bangkok-thonglor,https://avacasa.life/guides/bangkok-thonglor?u...,avacasa.life,1,0.287938,"[""Attraction Type Highlight Why It Matters Dis...",true_positive,Stored cleaned-main-content evidence visibly s...,stored_cleaned_main_content_html_or_markdown
4,table,detected_positive,https://baanlyy.com/areas/best-for/expats,https://baanlyy.com/areas/best-for/expats?utm_...,baanlyy.com,1,0.080581,"[""# Neighbourhood Expat-fit score Why it ranks...",true_positive,Stored cleaned-main-content evidence visibly s...,stored_cleaned_main_content_html_or_markdown
...,...,...,...,...,...,...,...,...,...,...,...
95,comparison,measured_negative,https://adamdecorcenter.com/vive-2,https://www.adamdecorcenter.com/vive-2/?utm_so...,adamdecorcenter.com,0,NaN,[],true_negative_in_stored_snapshot,No matching structure was found in the stored ...,stored_cleaned_main_content_html_or_markdown
96,comparison,measured_negative,https://insights.aecom.com/insights/article/lu...,https://insights.aecom.com/insights/article/lu...,aecom.com,0,NaN,[],true_negative_in_stored_snapshot,No matching structure was found in the stored ...,stored_cleaned_main_content_html_or_markdown
97,comparison,measured_negative,https://aestiqthonglor.com/,https://aestiqthonglor.com/?utm_source=chatgpt...,aestiqthonglor.com,0,NaN,[],true_negative_in_stored_snapshot,No matching structure was found in the stored ...,stored_cleaned_main_content_html_or_markdown
98,comparison,measured_negative,https://agent.in.th/read/buy-condo,https://agent.in.th/read/buy-condo?utm_source=...,agent.in.th,0,NaN,[],true_negative_in_stored_snapshot,No matching structure was found in the stored ...,stored_cleaned_main_content_html_or_markdown


,check,passed
0,table: ratios between 0 and 1,True
1,table: absence is not position zero,True
2,table: quartiles match ratios,True
3,table: counts non-negative,True
4,list: ratios between 0 and 1,True
5,list: absence is not position zero,True
6,list: quartiles match ratios,True
7,list: counts non-negative,True
8,faq: ratios between 0 and 1,True
9,faq: absence is not position zero,True


## 9. Model-readiness handoff

This is a recommendation table only. No new regression is estimated. Presence and position should remain separate in any later specification.

In [10]:
readiness = pd.read_csv(OUT / 'tables/position_feature_model_readiness.csv')
display(readiness)

,feature,recommended_representation,suggested_reference_group,full_sample_usable,conditional_sample_usable,domain_FE_usable,fixed_effect_readiness,sparse_cell_risk,multicollinearity_risk,recommended_model_role,reason
0,table,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,high,main model candidate,Coverage=15.8%; 33 domains vary in both positi...
1,list,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,high,main model candidate,Coverage=46.6%; 83 domains vary in both positi...
2,faq,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=13.3%; 32 domains vary in both positi...
3,direct_answer,"First quartile, middle half, last quartile",No feature for full sample; Q4 for feature-pre...,True,True,False,Weak within-domain variation,high,high,robustness check,Coverage=3.1%; 7 domains vary in both position...
4,definition,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=41.0%; 93 domains vary in both positi...
5,comparison,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=20.6%; 56 domains vary in both positi...
6,steps,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=14.0%; 46 domains vary in both positi...
7,numeric_evidence,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,high,main model candidate,Coverage=51.5%; 89 domains vary in both positi...
8,external_citation,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=29.8%; 74 domains vary in both positi...
9,question_heading,"Q1, Q2, Q3, Q4",No feature for full sample; Q4 for feature-pre...,True,True,True,Ready,low,moderate,position extension,Coverage=19.6%; 53 domains vary in both positi...


## 10. Evidence-based conclusion

The conclusion is generated directly from the validated diagnostics and is also exported as plain text for downstream review.

In [11]:
findings = (OUT / 'POSITION_FEATURE_EDA_FINDINGS.txt').read_text(encoding='utf-8')
print(findings)

POSITION FEATURE EDA FINDINGS

Scope
5,758 surfaced source-prompt rows; 2,881 unique URLs; 586 domains; citation rate 32.9%.

Coverage
Features with at least 10 percent eligible-page coverage: table, list, faq, definition, comparison, steps, numeric_evidence, external_citation, question_heading.

Descriptive citation differences
Largest absolute cited versus not-cited median-position differences: question_heading, list, table, external_citation, comparison. These are descriptive associations, not causal effects.

Domain fixed-effects readiness
Ready or usable with caution: table, list, faq, definition, comparison, steps, numeric_evidence, external_citation, question_heading.
Not suitable: none.

Future representation
Use the feature-specific grouping in position_feature_model_readiness.csv. No-feature and position effects must remain separate. Sparse groups must not be merged silently.

Future model role
Main-model candidates: table, list, numeric_evidence.
Position extensions: faq, de